# Fase 1: Ingesta con PySpark
## Carga desde Raw → Calidad → Quarantine → Processed

**Dataset:** `laptop_scrap_data.csv`  
**Origen:** `datalake/raw/`  
**Destinos:** `datalake/processed/` (datos limpios) y `datalake/quarantine/` (registros anómalos)

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import os

spark = SparkSession.builder \
    .appName("Ingesta_Laptops") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
spark

Spark version: 4.1.1


---
## 1. Lectura del dataset RAW

In [ ]:
RAW_PATH = "../datalake/raw/laptop_scrap_data.csv"
PROCESSED_PATH = "../datalake/processed/"
QUARANTINE_PATH = "../datalake/quarantine/"

df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .option("treatEmptyValuesAsNulls", "true") \
    .csv(RAW_PATH)

print(f"Filas cargadas: {df_raw.count():,}")
print(f"Columnas: {len(df_raw.columns)}")
print(f"Particiones: {df_raw.rdd.getNumPartitions()}")

Filas cargadas: 1,563
Columnas: 21
Particiones: 1


In [4]:
df_raw.printSchema()

root
 |-- Company: string (nullable = true)
 |-- TypeName: string (nullable = true)
 |-- Inches: double (nullable = true)
 |-- ScreenResolution: string (nullable = true)
 |-- Cpu: string (nullable = true)
 |-- Gpu: string (nullable = true)
 |-- OpSys: string (nullable = true)
 |-- TouchScreen: integer (nullable = true)
 |-- Ips: integer (nullable = true)
 |-- X_res: integer (nullable = true)
 |-- Y_res: integer (nullable = true)
 |-- ppi: double (nullable = true)
 |-- Dedicated_Gpu: integer (nullable = true)
 |-- Ram_GB: integer (nullable = true)
 |-- Weight_kg: double (nullable = true)
 |-- SSD: integer (nullable = true)
 |-- HHD: integer (nullable = true)
 |-- Storage_Type: string (nullable = true)
 |-- Total_Storage_GB: integer (nullable = true)
 |-- Storage_Category: string (nullable = true)
 |-- Price: double (nullable = true)



In [5]:
df_raw.show(5, truncate=False, vertical=True)

-RECORD 0---------------------------------------------------
 Company          | MSI                                     
 TypeName         | MSI Prestige 16 AI+                     
 Inches           | 16.0                                    
 ScreenResolution | 2880 x 1800                             
 Cpu              | Intel Core Ultra X7 358H                
 Gpu              | Intel Arc B390                          
 OpSys            | Windows                                 
 TouchScreen      | 0                                       
 Ips              | 0                                       
 X_res            | 2880                                    
 Y_res            | 1800                                    
 ppi              | 212.26                                  
 Dedicated_Gpu    | 1                                       
 Ram_GB           | 64                                      
 Weight_kg        | 1.59                                    
 SSD              | 1000

---
## 2. Diagnóstico rápido de calidad

In [6]:
# Perfil de nulos por columna
null_profile = df_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
])
null_profile.show(vertical=True)

-RECORD 0---------------
 Company          | 3   
 TypeName         | 3   
 Inches           | 3   
 ScreenResolution | 3   
 Cpu              | 3   
 Gpu              | 3   
 OpSys            | 3   
 TouchScreen      | 3   
 Ips              | 3   
 X_res            | 3   
 Y_res            | 3   
 ppi              | 3   
 Dedicated_Gpu    | 3   
 Ram_GB           | 3   
 Weight_kg        | 3   
 SSD              | 3   
 HHD              | 3   
 Storage_Type     | 3   
 Total_Storage_GB | 3   
 Storage_Category | 3   
 Price            | 3   



In [7]:
total_rows = df_raw.count()

null_counts = df_raw.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total_rows * 100).alias(c)
    for c in df_raw.columns
])

print("% de nulos por columna:")
null_counts.show(vertical=True)

% de nulos por columna:
-RECORD 0-------------------------------
 Company          | 0.19193857965451055 
 TypeName         | 0.19193857965451055 
 Inches           | 0.19193857965451055 
 ScreenResolution | 0.19193857965451055 
 Cpu              | 0.19193857965451055 
 Gpu              | 0.19193857965451055 
 OpSys            | 0.19193857965451055 
 TouchScreen      | 0.19193857965451055 
 Ips              | 0.19193857965451055 
 X_res            | 0.19193857965451055 
 Y_res            | 0.19193857965451055 
 ppi              | 0.19193857965451055 
 Dedicated_Gpu    | 0.19193857965451055 
 Ram_GB           | 0.19193857965451055 
 Weight_kg        | 0.19193857965451055 
 SSD              | 0.19193857965451055 
 HHD              | 0.19193857965451055 
 Storage_Type     | 0.19193857965451055 
 Total_Storage_GB | 0.19193857965451055 
 Storage_Category | 0.19193857965451055 
 Price            | 0.19193857965451055 



In [8]:
# Duplicados exactos
dup_count = df_raw.count() - df_raw.distinct().count()
print(f"Filas duplicadas exactas: {dup_count}")

Filas duplicadas exactas: 2


In [9]:
# Estadísticas descriptivas de columnas numéricas
numeric_cols = [c for c, t in df_raw.dtypes if t in ('int', 'bigint', 'double', 'float')]
print("Columnas numéricas:", numeric_cols)

df_raw.select(numeric_cols).summary().show(vertical=True, truncate=False)

Columnas numéricas: ['Inches', 'TouchScreen', 'Ips', 'X_res', 'Y_res', 'ppi', 'Dedicated_Gpu', 'Ram_GB', 'Weight_kg', 'SSD', 'HHD', 'Total_Storage_GB', 'Price']
-RECORD 0-------------------------------
 summary          | count               
 Inches           | 1560                
 TouchScreen      | 1560                
 Ips              | 1560                
 X_res            | 1560                
 Y_res            | 1560                
 ppi              | 1560                
 Dedicated_Gpu    | 1560                
 Ram_GB           | 1560                
 Weight_kg        | 1560                
 SSD              | 1560                
 HHD              | 1560                
 Total_Storage_GB | 1560                
 Price            | 1560                
-RECORD 1-------------------------------
 summary          | mean                
 Inches           | 15.311858974358811  
 TouchScreen      | 0.7262820512820513  
 Ips              | 0.7153846153846154  
 X_res            |

---
## 3. Aislamiento de registros anómalos → Quarantine

Criterios de anomalía:
- Filas con `Price` nulo o <= 0
- `Ram_GB` nulo o <= 0
- `Weight_kg` nulo o <= 0
- `Inches` nulo o <= 0

In [10]:
anomaly_condition = (
    F.col("Price").isNull() | (F.col("Price") <= 0) |
    F.col("Ram_GB").isNull() | (F.col("Ram_GB") <= 0) |
    F.col("Weight_kg").isNull() | (F.col("Weight_kg") <= 0) |
    F.col("Inches").isNull() | (F.col("Inches") <= 0)
)

df_quarantine = df_raw.filter(anomaly_condition)
df_clean = df_raw.filter(~anomaly_condition)

print(f"Registros en Quarantine: {df_quarantine.count():,}")
print(f"Registros Clean: {df_clean.count():,}")

Registros en Quarantine: 14
Registros Clean: 1,549


In [11]:
# Inspeccionar qué se fue a quarantine
df_quarantine.show(20, truncate=False)

+--------+----------------------------+------+----------------+---------------------+-----------------------------------+-------+-----------+----+-----+-----+------+-------------+------+---------+----+----+------------+----------------+-----------------+------+
|Company |TypeName                    |Inches|ScreenResolution|Cpu                  |Gpu                                |OpSys  |TouchScreen|Ips |X_res|Y_res|ppi   |Dedicated_Gpu|Ram_GB|Weight_kg|SSD |HHD |Storage_Type|Total_Storage_GB|Storage_Category |Price |
+--------+----------------------------+------+----------------+---------------------+-----------------------------------+-------+-----------+----+-----+-----+------+-------------+------+---------+----+----+------------+----------------+-----------------+------+
|AORUS   |AORUS MASTER 16 AM6J        |16.0  |2560 x 1600     |AMD Ryzen 9 9955HX3D |NVIDIA GeForce RTX 5090 (Laptop)   |Windows|0          |0   |2560 |1600 |188.68|1            |64    |0.0      |4000|0   |SSD Only

---
## 4. Estandarización de tipos de datos

Correcciones detectadas:
- `HHD` → renombrar a `HDD` (typo claro)
- `Ram_GB` → Integer
- `Weight_kg` → Double
- `TouchScreen`, `Ips`, `Dedicated_Gpu` → Integer (binarias)

In [12]:
df_clean = df_clean \
    .withColumnRenamed("HHD", "HDD") \
    .withColumn("Ram_GB", F.col("Ram_GB").cast("int")) \
    .withColumn("Weight_kg", F.col("Weight_kg").cast("double")) \
    .withColumn("Inches", F.col("Inches").cast("double")) \
    .withColumn("TouchScreen", F.col("TouchScreen").cast("int")) \
    .withColumn("Ips", F.col("Ips").cast("int")) \
    .withColumn("Dedicated_Gpu", F.col("Dedicated_Gpu").cast("int")) \
    .withColumn("Price", F.col("Price").cast("double")) \
    .withColumn("SSD", F.col("SSD").cast("int")) \
    .withColumn("Total_Storage_GB", F.col("Total_Storage_GB").cast("int"))

df_clean.printSchema()

root
 |-- Company: string (nullable = true)
 |-- TypeName: string (nullable = true)
 |-- Inches: double (nullable = true)
 |-- ScreenResolution: string (nullable = true)
 |-- Cpu: string (nullable = true)
 |-- Gpu: string (nullable = true)
 |-- OpSys: string (nullable = true)
 |-- TouchScreen: integer (nullable = true)
 |-- Ips: integer (nullable = true)
 |-- X_res: integer (nullable = true)
 |-- Y_res: integer (nullable = true)
 |-- ppi: double (nullable = true)
 |-- Dedicated_Gpu: integer (nullable = true)
 |-- Ram_GB: integer (nullable = true)
 |-- Weight_kg: double (nullable = true)
 |-- SSD: integer (nullable = true)
 |-- HDD: integer (nullable = true)
 |-- Storage_Type: string (nullable = true)
 |-- Total_Storage_GB: integer (nullable = true)
 |-- Storage_Category: string (nullable = true)
 |-- Price: double (nullable = true)



---
## 5. Escritura a Procesado (Parquet) y Quarantine (Parquet)

Se escribe en formato **Parquet** columnar con compresión snappy.

In [13]:
df_clean.write \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(PROCESSED_PATH)

df_quarantine.write \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(QUARANTINE_PATH)

print("✅ Datos escritos en processed/ y quarantine/")

Py4JJavaError: An error occurred while calling o594.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:314)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdir

In [ ]:
# Verificación de escritura
df_check_processed = spark.read.parquet(PROCESSED_PATH)
df_check_quarantine = spark.read.parquet(QUARANTINE_PATH)

print(f"Processed - Filas: {df_check_processed.count():,} | Columnas: {len(df_check_processed.columns)}")
print(f"Quarantine - Filas: {df_check_quarantine.count():,} | Columnas: {len(df_check_quarantine.columns)}")

print("\nColumnas en processed:", df_check_processed.columns)
df_check_processed.show(3, truncate=False, vertical=True)

In [ ]:
spark.stop()